In [ ]:
!unzip "/content/drive/My Drive/split.zip" -d "/content/dataset"

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/split/train/real/000000525013.jpg  
  inflating: /content/dataset/split/train/real/000000525043.jpg  
  inflating: /content/dataset/split/train/real/000000525122.jpg  
  inflating: /content/dataset/split/train/real/000000525156.jpg  
  inflating: /content/dataset/split/train/real/000000525159.jpg  
  inflating: /content/dataset/split/train/real/000000525257.jpg  
  inflating: /content/dataset/split/train/real/000000525279.jpg  
  inflating: /content/dataset/split/train/real/000000525291.jpg  
  inflating: /content/dataset/split/train/real/000000525330.jpg  
  inflating: /content/dataset/split/train/real/000000525343.jpg  
  inflating: /content/dataset/split/train/real/000000525366.jpg  
  inflating: /content/dataset/split/train/real/000000525403.jpg  
  inflating: /content/dataset/split/train/real/000000525410.jpg  
  inflating: /content/dataset/split/train/real/000000525454.jpg  
  inflating: /content/dat

In [ ]:
import tensorflow as tf
print(tf.__version__)  # Check if TensorFlow is available


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
!pip install tensorflow --upgrade



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 845.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 79.6 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.18.0
    Uninstalling tensorboard-2.18.0:
      Successfully uninstalled tensorboard-2.18.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.18.0
    Uninstalling tensorflow-2.18.0:
      Successfully uninstalled tensorflow-2.18.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.19.0

In [ ]:
import tensorflow as tf
print(tf.__version__)  # Check if TensorFlow is available


2.19.0


In [ ]:
import tensorflow as tf
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard

# Paths
DATASET_PATH = "/content/dataset/split"  # Update if needed
TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VAL_DIR = os.path.join(DATASET_PATH, "val")

# Image Parameters
IMG_SIZE = (512, 512)
BATCH_SIZE = 32

# No Augmentation, No Rescaling (since images are already normalized)
datagen = ImageDataGenerator()

train_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# Model Definition
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_tensor=Input(shape=(512, 512, 3)))
base_model.trainable = False  # Freeze base model initially

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.4)(x)
out = Dense(1, activation='sigmoid')(x)
model = Model(inputs=base_model.input, outputs=out)

model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_loss', mode='min'),
    TensorBoard(log_dir='./logs')
]

# Train Model
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks
)

# Save Model
model.save("final_model.h5")


Found 28536 images belonging to 2 classes.
Found 3566 images belonging to 2 classes.
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(None, 512, 512, 3))
  warnings.warn(msg)


  8/892 ━━━━━━━━━━━━━━━━━━━━ 7:48:24 32s/step - accuracy: 0.4379 - loss: 0.7852

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard

# ====== STEP 1: Initialize TPU ======
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # Detect TPU
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("✅ TPU Detected and Initialized!")
except:
    strategy = tf.distribute.get_strategy()  # Default to CPU/GPU
    print("⚠️ TPU Not Found, using default strategy.")

# ====== STEP 2: Paths ======
DATASET_PATH = "/content/dataset/split"  # Update as needed
TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VAL_DIR = os.path.join(DATASET_PATH, "val")

# ====== STEP 3: Image Parameters ======
IMG_SIZE = (380, 380)  # EfficientNetB3 requires 380x380
BATCH_SIZE = 32  # Adjust based on TPU memory

# ====== STEP 4: Data Preparation ======
datagen = ImageDataGenerator(rescale=1.0/255)  # No augmentation

train_generator = datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary'
)

val_generator = datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary'
)

# ====== STEP 5: Model Definition ======
with strategy.scope():  # Ensure TPU compatibility
    base_model = EfficientNetB3(weights='imagenet', include_top=False, input_tensor=Input(shape=(380, 380, 3)))
    base_model.trainable = False  # Freeze base model initially

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.4)(x)
    out = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=out)

    model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# ====== STEP 6: Callbacks ======
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True, monitor='val_loss', mode='min'),
    TensorBoard(log_dir='./logs')
]

# ====== STEP 7: Train Model ======
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks
)

# ====== STEP 8: Save Model ======
model.save("final_model.h5")
print("✅ Model training complete and saved!")


⚠️ TPU Not Found, using default strategy.
Found 28536 images belonging to 2 classes.
Found 3566 images belonging to 2 classes.
Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor_778']
Received: inputs=Tensor(shape=(None, 380, 380, 3))
  warnings.warn(msg)


  2/892 ━━━━━━━━━━━━━━━━━━━━ 9:56:30 40s/step - accuracy: 0.5312 - loss: 0.7061 

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()  # Detect TPU
    print("✅ TPU Found:", tpu.master())
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("✅ TPU Initialized Successfully!")
except ValueError:
    print("⚠️ TPU Not Found, using default strategy.")
    strategy = tf.distribute.get_strategy()


⚠️ TPU Not Found, using default strategy.


In [ ]:
import tensorflow as tf
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Check if GPU is available
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))


Num GPUs Available: 1


In [ ]:
IMG_SIZE = 380  # Match EfficientNetB3 input size
BATCH_SIZE = 32

train_dir = "/content/dataset/split/train"   # Update with your train directory
val_dir = "/content/dataset/split/val"       # Update with your validation directory

# No augmentation, only resizing
train_datagen = ImageDataGenerator()

val_datagen = ImageDataGenerator()  # No normalization for validation

# Load images
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)



Found 28536 images belonging to 2 classes.
Found 3566 images belonging to 2 classes.


In [ ]:
# Load EfficientNetB3 with pre-trained ImageNet weights
base_model = EfficientNetB3(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # Freeze base model initially

# Add custom classifier
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid')(x)  # Binary classification

# Create final model
model = Model(inputs=base_model.input, outputs=output)

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Model Ready for Training")


43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ Model Ready for Training


In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)

callbacks = [early_stopping, reduce_lr]


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Define callbacks
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6)

checkpoint = ModelCheckpoint(
    "best_model.h5",  # Saves only the best model
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

callbacks = [early_stopping, reduce_lr, checkpoint]  # Add ModelCheckpoint

# Train model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,  # Adjust as needed
    callbacks=callbacks  # Include callbacks
)

Epoch 1/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step - accuracy: 0.8202 - loss: 0.4582
Epoch 1: val_loss improved from inf to 0.32219, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 383s 430ms/step - accuracy: 0.8202 - loss: 0.4582 - val_accuracy: 0.8957 - val_loss: 0.3222 - learning_rate: 1.0000e-04
Epoch 2/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - accuracy: 0.8864 - loss: 0.3197
Epoch 2: val_loss improved from 0.32219 to 0.26688, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 326s 365ms/step - accuracy: 0.8865 - loss: 0.3197 - val_accuracy: 0.9103 - val_loss: 0.2669 - learning_rate: 1.0000e-04
Epoch 3/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step - accuracy: 0.9024 - loss: 0.2719
Epoch 3: val_loss improved from 0.26688 to 0.24118, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 321s 360ms/step - accuracy: 0.9024 - loss: 0.2719 - val_accuracy: 0.9136 - val_loss: 0.2412 - learning_rate: 1.0000e-04
Epoch 4/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - accuracy: 0.9076 - loss: 0.2509
Epoch 4: val_loss improved from 0.24118 to 0.22464, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 320s 359ms/step - accuracy: 0.9076 - loss: 0.2509 - val_accuracy: 0.9190 - val_loss: 0.2246 - learning_rate: 1.0000e-04
Epoch 5/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step - accuracy: 0.9141 - loss: 0.2310
Epoch 5: val_loss improved from 0.22464 to 0.21495, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 313s 350ms/step - accuracy: 0.9141 - loss: 0.2310 - val_accuracy: 0.9206 - val_loss: 0.2150 - learning_rate: 1.0000e-04
Epoch 6/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step - accuracy: 0.9143 - loss: 0.2278
Epoch 6: val_loss improved from 0.21495 to 0.20650, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 303s 339ms/step - accuracy: 0.9143 - loss: 0.2278 - val_accuracy: 0.9229 - val_loss: 0.2065 - learning_rate: 1.0000e-04
Epoch 7/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step - accuracy: 0.9157 - loss: 0.2190
Epoch 7: val_loss improved from 0.20650 to 0.20107, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 316s 333ms/step - accuracy: 0.9157 - loss: 0.2190 - val_accuracy: 0.9246 - val_loss: 0.2011 - learning_rate: 1.0000e-04
Epoch 8/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.9179 - loss: 0.2154
Epoch 8: val_loss improved from 0.20107 to 0.19708, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 293s 329ms/step - accuracy: 0.9179 - loss: 0.2154 - val_accuracy: 0.9257 - val_loss: 0.1971 - learning_rate: 1.0000e-04
Epoch 9/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step - accuracy: 0.9183 - loss: 0.2105
Epoch 9: val_loss improved from 0.19708 to 0.19399, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 294s 330ms/step - accuracy: 0.9183 - loss: 0.2105 - val_accuracy: 0.9262 - val_loss: 0.1940 - learning_rate: 1.0000e-04
Epoch 10/10
892/892 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step - accuracy: 0.9195 - loss: 0.2095
Epoch 10: val_loss improved from 0.19399 to 0.19064, saving model to best_model.h5


892/892 ━━━━━━━━━━━━━━━━━━━━ 294s 330ms/step - accuracy: 0.9195 - loss: 0.2095 - val_accuracy: 0.9262 - val_loss: 0.1906 - learning_rate: 1.0000e-04


In [ ]:
model.save("best_model.keras")

In [ ]:
from tensorflow.keras.models import load_model

# Load the best model
model = load_model("best_model.keras")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define test data directory
test_dir = "/content/dataset/split/test"

# Create ImageDataGenerator for testing (no augmentation, just rescaling)
test_datagen = ImageDataGenerator()

# Load images from test directory
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(380, 380),  # Adjust based on your model's input size
    batch_size=32,  # Change batch size as needed
    class_mode="binary",  # Use 'binary' for binary classification
    shuffle=False  # No shuffling for evaluation
)


Found 3568 images belonging to 2 classes.


In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}")


112/112 ━━━━━━━━━━━━━━━━━━━━ 51s 342ms/step - accuracy: 0.9161 - loss: 0.1909
Test Accuracy: 0.9386, Test Loss: 0.1713
